# Analytics Q&A Agent — Demo

A short walkthrough of the five question types the agent handles:
1. Simple aggregate (DAU)
2. Filtered aggregate (US users only)
3. Cohort / D7 retention
4. Week-over-week comparison
5. Ambiguous question → clarification (not an answer)

Plus a bonus chart cell at the end.

**Setup checklist** before running:
- `.env` has `ANTHROPIC_API_KEY=sk-ant-...`
- `analytics.db` exists (the setup cell below generates it if missing)
- `pip install -r requirements.txt`

In [ ]:
# Setup — load env, make the project package importable, ensure DB exists.
import os, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')

from agent.agent import ask, Answer, ClarificationNeeded

if not (ROOT / 'analytics.db').exists():
    import data.generate_db as gen
    gen.main()

print('API key loaded:', bool(os.getenv('ANTHROPIC_API_KEY')))
print('DB exists:     ', (ROOT / 'analytics.db').exists())

## 1. Simple aggregate — DAU last week

In [ ]:
r = ask('What was our DAU last week?')
assert isinstance(r, Answer)
print('SUMMARY:', r.summary)
print('SELF-CHECK ok=', r.self_check.ok, '|', r.self_check.reason)
print('SQL:', r.sql)
r.df

## 2. Filtered aggregate — US users only

In [ ]:
r = ask('DAU for US users last week')
print('SUMMARY:', r.summary)
print('SQL:', r.sql)
r.df

## 3. Cohort — D7 retention (hardest shape)

In [ ]:
r = ask('What is D7 retention for users who signed up 30 days ago?')
print('SUMMARY:', r.summary)
print('SQL:', r.sql)
r.df

## 4. Comparison — this week vs last

In [ ]:
r = ask('How did revenue this week compare to last week?')
print('SUMMARY:', r.summary)
print('SQL:', r.sql)
r.df

## 5. Ambiguous — the agent should clarify, not guess

Expected: `ask()` returns a `ClarificationNeeded`, not an `Answer`.

In [ ]:
r = ask('Show me active users')
assert isinstance(r, ClarificationNeeded), f'expected clarification, got {type(r).__name__}'
print('CLARIFY:', r.clarification)

## Bonus — a chart

For multi-row results, `ask()` returns a matplotlib `Figure` in `r.chart`.
Scalar results (single row) return `r.chart = None` — by design.

In [ ]:
r = ask('Plot DAU for the last 14 days')
if isinstance(r, Answer):
    print(r.summary)
r.chart if isinstance(r, Answer) else r.clarification